In [1]:
%pip install gdown tensorboard zarr earthaccess folium
%pip install terratorch==1.1.1

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import os

import gdown

In [9]:

hwds_google_drive_id = '1c5dQxAYfv4b4DCLyrjP0XyTERiofdHF0'
drive_url = f'https://drive.google.com/uc?id={hwds_google_drive_id}'
filename = '0095_S30'

if not Path(filename).exists():
  gdown.download(drive_url, f'{filename}.zip', quiet=False)
  !unzip {filename}.zip -d {filename}

Downloading...
From (original): https://drive.google.com/uc?id=1c5dQxAYfv4b4DCLyrjP0XyTERiofdHF0
From (redirected): https://drive.google.com/uc?id=1c5dQxAYfv4b4DCLyrjP0XyTERiofdHF0&confirm=t&uuid=e4e5fc7f-c5e5-43be-842b-5587ae5d5328
To: /home/wbhorn/repositories/tools/terramind-docs/severe-weather/notebooks/0095_S30.zip
100%|██████████| 342M/342M [00:08<00:00, 42.4MB/s] 


Archive:  0095_S30.zip
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.B02.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.B03.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.B04.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.B06.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.B07.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.B08.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.BROWSE.png  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.EVENT.tif  
  inflating: 0095_S30/0095.HLS.S30.2018178.v2.0.Fmask.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.B02.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.B03.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.B04.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.B06.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.B07.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.B08.tif  
  inflating: 0095_S30/0095.HLS.S30.2018183.v2.0.BROWSE.png  
  inflating: 0095_S30/0095.HLS.S30.2018

In [ ]:
from typing import Any
from torchgeo.datasets import RasterDataset, stack_samples
import matplotlib.pyplot as plt

class HWDSHLSDataset(RasterDataset):
    """
    Custom HWDS HLS dataset using TorchGeo's RasterDataset base class.

    Example file name:
        0095.HLS.S30.2018178.v2.0.B02.tif

    Bands:
        B01 ->  COASTAL
        B02 ->  BLUE
        B03 ->  GREEN
        B04 ->  RED
        B05 ->  REDEDGE1
        B06 ->  REDEDGE2
        B07 ->  REDEDGE3
        B08 ->  NIR
        B8A ->  NIR08
        B09 ->  NIR09
        B11 ->  SWIR16
        B12 ->  SWIR22
        EVENT -> Damage Label
        Fmask -> Cloud Mask
    """

    filename_glob = "*HLS.*.tif"
    filename_regex = r"^(?P<swathID>\d+)\.HLS\.(?P<sensor>S)30\.(?P<date>\d+)\.v(?P<version>\d+\.\d+)\.(?P<band>B\d+|EVENT|Fmask)\.tif$"
    date_format = '%Y%j'

    separate_files = True
    is_image = True

    all_bands = ("B02", "B03", "B04", "B06", "B07", "B08", "EVENT", "Fmask")
    rgb_bands = ('B04', 'B03', 'B02')

    def plot(
            self,
            sample: dict[str, Any],
            show_titles: bool = True,
            suptitle: str | None = None,
        ):
        rgb_indices = []
        for band in self.rgb_bands:
            rgb_indices.append(self.bands.index(band))

        image = sample['image'][rgb_indices].permute(1, 2, 0)
        # DN = 10000 * REFLECTANCE
        # https://docs.sentinel-hub.com/api/latest/data/sentinel-2-l2a/
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))

        ax.imshow(image)
        ax.axis('off')

        if show_titles:
            ax.set_title('Image')

        if suptitle is not None:
            plt.suptitle(suptitle)

        return fig

In [ ]:
from torch.utils.data import DataLoader
from torchgeo.samplers import RandomGeoSampler

patch_size = 224
dataset = HWDSHLSDataset(paths=filename)

sampler = RandomGeoSampler(
    dataset,
    size=patch_size,
    length=10,
)

dataloader = DataLoader(
    dataset,
    sampler=sampler,
    batch_size=2,
    collate_fn=stack_samples,
)